# MLP khôi phục cột bị che

Xem việc học khôi phục giá trị trước khi nhìn nhãn có cải thiện MLP phân loại gian lận hay không.

So sánh cấu hình `mlp-masked` với `mlp-scratch` trên RTX 4060.

## 1. Chọn chế độ

- `view` đọc kết quả đã lưu. `VIEW_RESULTS` chọn thư mục `full` hoặc `smoke`.
- `smoke` kiểm tra luồng chạy với dữ liệu rút gọn, 2 fold, 1 epoch tiền huấn luyện và 1 epoch phân loại. Đối chứng chỉ chạy 1 epoch phân loại.
- `full` dùng 699.960 dòng train, 299.984 dòng test và 5 fold. Trình chạy tiếp tục từ các fold có kết quả hoàn chỉnh với cùng cấu hình.

Khi chạy huấn luyện, chọn kernel của môi trường PyTorch CUDA đã chuẩn bị cho NVIDIA RTX 4060.

In [1]:
from pathlib import Path

METHOD = 'mlp-masked'
CONTROL = 'mlp-scratch'
RUN_MODE = "view"  # "view", "smoke" hoặc "full"
VIEW_RESULTS = "full"  # Đổi thành "smoke" để xem một lượt chạy thử

if RUN_MODE not in {"view", "smoke", "full"}:
    raise ValueError("RUN_MODE phải là view, smoke hoặc full.")
if VIEW_RESULTS not in {"smoke", "full"}:
    raise ValueError("VIEW_RESULTS phải là smoke hoặc full.")

current = Path.cwd().resolve()
for candidate in (current, *current.parents):
    if (candidate / "s6e11_gpu" / "pretraining").is_dir():
        project_root = candidate
        break
else:
    raise RuntimeError("Mở notebook trong thư mục dự án hoặc một thư mục con.")

experiment_dir = project_root / "s6e11_gpu" / "pretraining"
runner = experiment_dir / "run_pretraining.py"
result_root = experiment_dir / "results" / (
    VIEW_RESULTS if RUN_MODE == "view" else RUN_MODE
)
print(f"Phương pháp: {METHOD} | Đối chứng: {CONTROL} | Kết quả: {result_root.name}")

Phương pháp: mlp-masked | Đối chứng: mlp-scratch | Kết quả: full


## 2. Dữ liệu và phạm vi của fold

- Dùng 24 đặc trưng của S6E11, gồm 13 cột số, 5 cột phân loại và 6 cờ. `id` dùng để ghép kết quả, còn `is_fraud` là nhãn phân loại.
- Lượt đầy đủ giữ cùng 5 fold phân tầng với mô hình đối chứng trong dự án. Encoder được học riêng từ phần train của từng fold. Các dòng validation và test được biến đổi bằng cấu hình đã học từ phần train.
- Giai đoạn tiền huấn luyện học từ các cột đầu vào. Giai đoạn tinh chỉnh học từ nhãn `is_fraud` của phần train. Validation ngoài fold dùng để chấm điểm sau khi huấn luyện.

## 3. Tiền huấn luyện

MLP biến đặc trưng số và mã phân loại thành một vector biểu diễn 64 chiều cho từng giao dịch.

1. Chọn **đúng 7 trong 24 cột** của mỗi giao dịch, tương ứng 30% sau khi làm tròn. Mạng nhận tín hiệu cho biết vị trí bị che.
2. Khôi phục giá trị chỉ ở các vị trí đã che. Cột số dùng Smooth L1, còn cột phân loại và cờ dùng cross-entropy.
3. Giữ phần tạo biểu diễn để huấn luyện đầu phân loại gian lận.

[VIME, NeurIPS 2020](https://papers.nips.cc/paper/2020/hash/7d97667a3e056acab9aaf653807b4a03-Abstract.html) · [Mã mô hình](../neural_models.py)

## 4. Tinh chỉnh và đối chứng

Lượt đầy đủ cố định **3 epoch tiền huấn luyện** và **6 epoch phân loại**. Batch size là **4.096** trong bước tiền huấn luyện và **4.096** trong bước phân loại.

Đầu phân loại dự đoán một điểm gian lận cho mỗi dòng. Toàn bộ mạng được cập nhật bằng AdamW với tốc độ học 0,001, tối ưu binary cross-entropy (BCE) trên nhãn `is_fraud`. Đối chứng `mlp-scratch` khởi tạo ngẫu nhiên với cùng kiến trúc và chạy đúng 6 epoch phân loại trên cùng phép chia fold.

## 5. Chạy trên GPU

Chế độ huấn luyện dùng NVIDIA RTX 4060 qua `cuda:0`. Cell dưới khởi chạy hai cấu hình và in tiến độ từng fold. Chế độ `view` chỉ đọc kết quả ở cell tiếp theo.

In [2]:
if RUN_MODE == "view":
    print("Chế độ xem kết quả đã lưu.")
else:
    import os
    import subprocess
    import sys
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError("Hãy chọn kernel của môi trường PyTorch có CUDA.")
    print(f"Thiết bị: {torch.cuda.get_device_name(0)}", flush=True)
    command = [
        sys.executable, "-u", str(runner),
        "--variants", CONTROL, METHOD,
    ]
    if RUN_MODE == "smoke":
        command.append("--smoke")
    environment = dict(os.environ, PYTHONIOENCODING="utf-8")
    with subprocess.Popen(
        command, cwd=project_root, env=environment, text=True,
        encoding="utf-8", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

Chế độ xem kết quả đã lưu.


## 6. Đọc kết quả đã lưu

**OOF ROC AUC** đo khả năng xếp hạng trên toàn bộ dự đoán ngoài fold. Mỗi dòng được chấm bởi một mô hình học từ các phần khác. Bảng dưới cũng hiển thị **Average Precision**, **Log-loss** và chênh lệch AUC giữa hai cấu hình trên từng fold khi có đủ dữ liệu.

In [3]:
import json
import pandas as pd
from IPython.display import display

records = {}
for variant in (CONTROL, METHOD):
    metric_path = result_root / variant / "metrics.json"
    if metric_path.is_file():
        with metric_path.open(encoding="utf-8") as file:
            records[variant] = json.load(file)
    else:
        print(f"Chưa có kết quả hoàn chỉnh của {variant} trong {result_root.name}.")

if records:
    rows = [
        {"Cấu hình": variant, "OOF ROC AUC": record["oof_auc"],
         "Average Precision": record.get("average_precision"),
         "Log-loss": record.get("log_loss")}
        for variant, record in records.items()
    ]
    display(pd.DataFrame(rows).set_index("Cấu hình").round(6))

    fold_tables = {}
    for variant, record in records.items():
        fold_rows = pd.DataFrame(record.get("folds", []))
        if {"fold", "auc"} <= set(fold_rows.columns):
            fold_tables[variant] = fold_rows.set_index("fold")["auc"]
    if fold_tables:
        by_fold = pd.DataFrame(fold_tables).sort_index()
        if {CONTROL, METHOD} <= set(by_fold.columns):
            by_fold["Δ tiền huấn luyện"] = by_fold[METHOD] - by_fold[CONTROL]
        display(by_fold.round(6))

    if {CONTROL, METHOD} <= records.keys():
        delta = records[METHOD]["oof_auc"] - records[CONTROL]["oof_auc"]
        print(f"Δ OOF ROC AUC so với {CONTROL}: {delta:+.6f}")

    timing_rows = []
    for variant, record in records.items():
        for fold in record.get("folds", []):
            timing_rows.append({
                "Cấu hình": variant, "Fold": fold.get("fold"),
                "Tiền huấn luyện (s)": fold.get("pretrain_seconds"),
                "Tinh chỉnh (s)": fold.get("finetune_seconds"),
                "Thiết bị": fold.get("device"),
            })
    if timing_rows:
        display(pd.DataFrame(timing_rows).round(1))

Δ OOF ROC AUC so với mlp-scratch: +0.000315


,OOF ROC AUC,Average Precision,Log-loss
Cấu hình,,,
mlp-scratch,0.941240,0.488468,0.047232
mlp-masked,0.941555,0.489459,0.047121


,mlp-scratch,mlp-masked,Δ tiền huấn luyện
fold,,,
1,0.940926,0.940917,-0.000008
2,0.942851,0.943175,0.000324
3,0.946039,0.946652,0.000613
4,0.941767,0.942496,0.000730
5,0.936464,0.936866,0.000402


,Cấu hình,Fold,Tiền huấn luyện (s),Tinh chỉnh (s),Thiết bị
0,mlp-scratch,1,0.0,2.3,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
1,mlp-scratch,2,0.0,2.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
2,mlp-scratch,3,0.0,2.1,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
3,mlp-scratch,4,0.0,2.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
4,mlp-scratch,5,0.0,6.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
5,mlp-masked,1,2.1,2.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
6,mlp-masked,2,2.0,1.8,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
7,mlp-masked,3,2.3,2.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
8,mlp-masked,4,2.4,2.0,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU
9,mlp-masked,5,7.1,5.8,cuda:0: NVIDIA GeForce RTX 4060 Laptop GPU


## 7. Diễn giải và kết hợp

So với cùng MLP khởi tạo ngẫu nhiên để quy phần chênh lệch về giai đoạn học khôi phục. Với hướng bổ sung đặc trưng, giữ encoder ở trạng thái sau tiền huấn luyện và lấy vector trước đầu phân loại.

Nếu thử ghép đặc trưng, chỉ dùng encoder tự giám sát đã đóng băng của chính fold đó để bổ sung vector cho `XGB-D03`. Nếu gộp điểm dự đoán, đối chiếu phương án cố định 95% `FINAL` + 5% mô hình mới với `FINAL` trên cùng các fold.

[Tổng hợp kết quả các hướng](../results/summary.md)